# 06 - Colab TUI feel: /cept CLI first, /cept agent second

## Objective

Type only `/cept` and keep working: first through the complete public CLI basics, then through the `/cept` agent with your own model credential.

## How this lesson runs

- Part A needs no secret. It drives the installed `cept` launcher directly; output is streamed and a nonzero exit raises immediately.
- Part B targets Google Colab (Linux). It installs OpenCode, fetches the teaching command, and chats through a `/cept >` prompt. Every turn saves files under `chat/`.
- A skipped cell is a decision, not a failure: it prints why it skipped and executes nothing.

## Claim boundary

`WORKFLOW_VALIDATED` only. A passing lesson never validates a physical project, field installation, or PowerFactory agreement.


In [ ]:
import hashlib
import importlib.util
import json
import os
import platform
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from pathlib import Path

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments):
    display = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output)
    return json.loads(output) if output.strip().startswith('{') else output
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))
def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [ ]:
capabilities = run_cli('capability', 'show')
doctor = run_cli('system', 'doctor')
show_table(['study type', 'engine', 'status'], [(study, details['engine'], details['status']) for study, details in sorted(capabilities['capabilities'].items())])
show_table(['doctor field', 'actual value'], [('status', doctor['status']), ('engine', doctor['engine']), ('engine version', doctor['engine_version'])])
assert doctor['status'] == 'PASS'
assert doctor['engine'] == 'opendss'
assert set(capabilities['capabilities']) == {'fault', 'hosting_capacity', 'load_flow', 'unbalanced_load_flow'}
assert all(details['status'] == 'candidate' for details in capabilities['capabilities'].values())


$ cept capability show


{


  "capabilities": {


    "fault": {


      "engine": "opendss",


...

      "status": "not_supported_initially"


    }


  }


}


| study type | engine | status |
| --- | --- | --- |
| fault | opendss | candidate |
| hosting_capacity | opendss | candidate |
| load_flow | opendss | candidate |
| unbalanced_load_flow | opendss | candidate |
| doctor field | actual value |
| --- | --- |
| status | PASS |
| engine | opendss |
| engine version | DSS C-API Library version 0.14.5 revision 87d85c2622c8281b92255335bc7c09b11191b21d based on OpenDSS SVN 3723 [FPC 3.2.2] (64-bit build) MVMULT INCREMENTAL_Y CONTEXT_API PM 20240329033747; License Status: Open 
DSS-Python version: 0.15.7
OpenDSSDirect.py version: 0.9.4 |


## Part A - /cept CLI basics

The public CLI has three nouns. This tour touches every verb once:

- `cept capability show` and `cept system doctor`
- `cept study demo ...` plus `cept study verify ...` for all four studies

Lessons 01-04 explain each study in depth; here we prove the whole surface works.


In [ ]:
TOUR = [("load-flow", "load_flow"), ("unbalanced-load-flow", "unbalanced_load_flow"), ("hosting-capacity", "hosting_capacity"), ("fault", "fault")]
rows = []
for demo_name, study_type in TOUR:
    out = Path.cwd() / 'runs' / f'06-{demo_name}'
    summary = run_cli('study', 'demo', demo_name, '--network', 'ieee13', '--out', out, '--force')
    verification = run_cli('study', 'verify', out)
    assert summary['status'] == 'PASS'
    assert verification['passed'] is True
    assert verification['study_type'] == study_type
    rows.append((demo_name, verification['status'], verification['claim'], summary['case_fingerprint'][:12]))
show_table(['demo', 'status', 'claim', 'case fingerprint'], rows)


$ cept study demo load-flow --network ieee13 --out '<notebook-workspace>\runs\06-load-flow' --force


{


  "status": "PASS",


  "claim": "WORKFLOW_VALIDATED",


  "study_type": "load_flow",


...

  "schema": "cept-public-verification-v1",


  "status": "PASS",


  "study_type": "fault"


}


| demo | status | claim | case fingerprint |
| --- | --- | --- | --- |
| load-flow | PASS | WORKFLOW_VALIDATED | 748c8026c9d6 |
| unbalanced-load-flow | PASS | WORKFLOW_VALIDATED | 4716c9c07f02 |
| hosting-capacity | PASS | WORKFLOW_VALIDATED | 849d2148e0b1 |
| fault | PASS | WORKFLOW_VALIDATED | 44d3766e5e8c |


## Part B - /cept as an agent (optional, bring your own key)

The agent is the same installed CLI with a narrator: it may only run `cept` commands and read `manifest.json` plus `public-verification.json`. The model transcript never counts as engineering evidence.

### How to start

- Run the wiring cell once (you do not need to read it).
- Then run `STATUS = cept_init_colab()`. It installs OpenCode on Colab, fetches the teaching command, and walks you through the key: open opencode.ai (or your provider), sign up / log in, open the API keys page, copy the key, and add it to Colab Secrets.
- The init cell prints only present or missing, never the value. Without a key every chat cell skips with a reason.


In [ ]:
import getpass

OPENCODE_BIN = Path.home() / ".opencode" / "bin" / "opencode"
KEY_NAMES = ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY')
COMMAND_URL = 'https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/agent/.opencode/commands/cept.md'

def _load_secret(names):
    for name in names:
        if os.environ.get(name, '').strip():
            return name, 'environment'
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        for name in names:
            try:
                value = (userdata.get(name) or '').strip()
            except Exception:
                continue
            if value:
                os.environ[name] = value
                return name, 'colab-secrets'
    try:
        pasted = getpass.getpass('Paste one provider key (hidden, memory-only; Enter to skip): ')
    except Exception:  # nbconvert raises StdinNotImplementedError here; any input failure means skip
        pasted = ''
    if pasted.strip():
        os.environ[KEY_NAMES[0]] = pasted.strip()
        return KEY_NAMES[0], 'typed-once'
    return '', ''

def _agent_ready():
    if platform.system() != 'Linux':
        return False, 'this step targets Google Colab (Linux)'
    if not OPENCODE_BIN.is_file():
        return False, 'opencode binary is missing; rerun the init cell'
    if not any(os.environ.get(name, '').strip() for name in KEY_NAMES):
        return False, 'model credential is missing; complete the init cell first'
    return True, 'ready'

def chat_turn(prompt, turn_id, model=''):
    turn_dir = Path.cwd() / 'chat' / turn_id
    turn_dir.mkdir(parents=True, exist_ok=True)
    (turn_dir / 'prompt.txt').write_text(prompt, encoding='utf-8')
    ready, reason = _agent_ready()
    if not ready:
        note = f'Skipped turn {turn_id}: {reason}. Nothing was executed.'
        print(note)
        (turn_dir / 'skipped.txt').write_text(note, encoding='utf-8')
        return None
    display = 'opencode run --command cept' + (f' --model {model}' if model else '')
    command = [str(OPENCODE_BIN), 'run', '--command', 'cept']
    if model:
        command += ['--model', model]
    command += [prompt]
    print('$ ' + display, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd(), encoding='utf-8', errors='replace')
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    (turn_dir / 'transcript.txt').write_text(output, encoding='utf-8')
    (turn_dir / 'record.json').write_text(json.dumps({'display': display, 'returncode': returncode}, indent=2), encoding='utf-8')
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['opencode', 'run', '--command', 'cept'], output=output)
    return output

def cept_chat(model=''):
    print('Type at the /cept > prompt; type exit to stop.')
    while True:
        try:
            question = input('/cept > ')
        except (EOFError, KeyboardInterrupt):
            print('ended (non-interactive run).')
            break
        if question.strip().lower() in ('exit', 'quit', '/exit'):
            break
        if not question.strip():
            continue
        chat_dir = Path.cwd() / 'chat'
        turn_id = f'{len(sorted(chat_dir.glob("turn-*"))) + 1:02d}' if chat_dir.exists() else '03'
        chat_turn(question, f'turn-{turn_id}', model=model)

def cept_init_colab():
    print('== Step 1/3: opencode binary ==')
    if OPENCODE_BIN.is_file():
        print('opencode already installed')
    elif platform.system() != 'Linux':
        print('Skipping opencode install: this step targets Google Colab (Linux).')
    else:
        subprocess.run('curl -fsSL https://opencode.ai/install | bash', shell=True, check=True)
        print(f'opencode installed: {OPENCODE_BIN.is_file()}')
    print('== Step 2/3: teaching command ==')
    try:
        command_dir = Path.cwd() / '.opencode' / 'commands'
        command_dir.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COMMAND_URL, command_dir / 'cept.md')
        fetched = (command_dir / 'cept.md').read_bytes()
        print(f'teaching command ready: sha256={hashlib.sha256(fetched).hexdigest()[:16]}')
    except Exception as exc:
        print(f'could not fetch teaching command: {exc}')
    print('== Step 3/3: model credential (memory only, never printed or saved) ==')
    print('Get a key first:')
    print('  1. Open opencode.ai (or your model provider) and sign up / log in.')
    print('  2. Open the API keys page, create a key, and copy it.')
    print('  3. In Colab: Secrets panel (key icon) -> add OPENAI_API_KEY or ANTHROPIC_API_KEY -> enable notebook access.')
    key_name, key_source = _load_secret(KEY_NAMES)
    if key_source:
        print(f'{key_name}: present (from {key_source})')
    else:
        print('model credential: missing — chat cells will skip until a key is provided. Rerun this cell after adding it.')
    return {'opencode': OPENCODE_BIN.is_file(), 'key': key_name if key_source else '', 'key_source': key_source}


In [ ]:
STATUS = cept_init_colab()


== Step 1/3: opencode binary ==
Skipping opencode install: this step targets Google Colab (Linux).
== Step 2/3: teaching command ==


teaching command ready: sha256=c09fd394db1e2e3a
== Step 3/3: model credential (memory only, never printed or saved) ==
Get a key first:
  1. Open opencode.ai (or your model provider) and sign up / log in.
  2. Open the API keys page, create a key, and copy it.
  3. In Colab: Secrets panel (key icon) -> add OPENAI_API_KEY or ANTHROPIC_API_KEY -> enable notebook access.
model credential: missing — chat cells will skip until a key is provided. Rerun this cell after adding it.


In [ ]:
PROMPT_01 = "Run the bounded IEEE 13-node load-flow demonstration and explain only verified artifacts."
chat_turn(PROMPT_01, "01")


Skipped turn 01: this step targets Google Colab (Linux). Nothing was executed.


In [ ]:
PROMPT_02 = "What does passed=true prove here, and what does it not prove?"
chat_turn(PROMPT_02, "02")


Skipped turn 02: this step targets Google Colab (Linux). Nothing was executed.


## Interpretation

Part A proves the CLI surface end to end. Part B proves an agent can drive that same surface without inventing values: every claim still traces to a solver artifact on disk.

## Exercise

Restart the kernel and run all cells. Then, with a key configured, rerun Part B and compare your `chat/` turns with the sanitized `SANITIZED_OBSERVE_REPLAY` in `public/agent/observe-replay.json`.
